# Beijing PM2.5 — Feature Engineering Pipeline (Corrected Dataset)

**Competition:** `inter-uni-datathon-stream-2-beijing-multi-site-air-quality`
**Task:** Forecast `PM2_5_next_hour` one hour ahead across 12 monitoring stations.
**Purpose:** Model-agnostic feature engineering. Run once, get parquet files for any model.

---

## ⚠️ READ THIS FIRST — what changed

The organizers removed **`current_PM2_5`** from both `train.csv` and `test.csv` because of
confirmed target leakage:

```
PM2_5_next_hour(t) == current_PM2_5(t+1)
```

**Consequence:** every feature in the previous pipeline that was derived from `current_PM2_5`
— its lags, changes, rolling stats, network aggregates, upwind values, and all `PM25_x_*`
interactions — **cannot be rebuilt**. The source column does not exist.

**What this notebook does instead:** rebuilds the same *conceptual families* (temporal,
spatial, rate, interaction) on the variables that actually remain, primarily **PM10** and
**CO** — the two with the strongest genuine, non-leaked signal:

| Check | PM10 | CO |
|---|---|---|
| Correlation with target | 0.866 | 0.778 |
| Next-row exact-match with target (leak test) | 0.232 | 0.000003 |
| Cross-station correlation (median) | 0.833 | 0.796 |

PM10's 0.232 exact-match rate is **not** a leak — it reflects the known `PM10 == PM2.5`
sensor co-occurrence (~23% of rows), not target reconstruction. A real leak looks like ~1.0.
All ten remaining columns were audited; none show leak-like behaviour.

### Do NOT reintroduce the leak
Do not create a `current_PM2_5` proxy by lagging the target
(`train.groupby("station")["PM2_5_next_hour"].shift(1)`). It is exact on train, **impossible
on test** (test has no target column), and recreates precisely what the organizers removed.
Any CV gain from it is a mirage.

---

## What this notebook produces

1. `features_train.parquet` — engineered training features (no target)
2. `features_test.parquet` — same feature set for test rows
3. `y.parquet` — target series, index-aligned to `features_train.parquet`
4. `feature_groups.json` — feature family → column list mapping, **for ablation**

This notebook deliberately **over-generates**. Selection is downstream: use
`feature_groups.json` to ablate by family rather than guessing which column is which.

---

## Feature families generated

| Family | Base variables | Notes |
|---|---|---|
| `RAW` | all pollutants + weather | PM10, SO2, NO2, CO, O3, TEMP, PRES, DEWP, RAIN, WSPM |
| `CALENDAR` | timestamp | year/month/day/hour/dayofweek + cyclical sin/cos |
| `WIND_ENCODING` | wd, WSPM | wd_sin/cos, wind vector components |
| `MISSINGNESS` | all measurement cols | one flag per column |
| `QUALITY_FLAGS` | O3, CO | sensor floor/ceiling/extreme flags |
| `PHYSICAL` | TEMP, DEWP, pollutant ratios | dew point depression, ratios |
| `LOCAL_LAGS_SHORT` | PM10, CO, others | 1h, 2h, 3h |
| `LOCAL_LAGS_MEDIUM` | PM10, CO, others | 6h, 12h |
| `LOCAL_LAGS_LONG` | PM10, CO, others | 24h, 48h |
| `LOCAL_ROLLING` | PM10, CO | rolling mean/std/min/max over 3/6/12/24h |
| `LOCAL_RATES` | PM10, CO | changes, pct changes, second differences |
| `NETWORK_STATE` | PM10, CO, NO2, SO2, O3 | same-timestamp cross-station aggregates |
| `NETWORK_HISTORY` | PM10, CO | lagged network aggregates |
| `NETWORK_RATES` | PM10, CO | regional change + acceleration |
| `SPATIAL_GRADIENTS` | all pollutants | local minus regional mean |
| `LOCAL_VS_NETWORK` | PM10, CO | divergence of local from regional dynamics |
| `UPWIND` | PM10, CO + geometry | directional transport features |
| `INTERACTIONS` | various | pollutant×weather, wind×regional |

---

## Validation results so far (LightGBM, holdouts A/B)

Holdout A = train pre-Sep 2015, validate Sep 2015–Feb 2016 (seasonal analogue).
Holdout B = train pre-Jul 2016, validate Jul–Aug 2016 (recency).

| Feature set | A | B | Kaggle |
|---|---|---|---|
| Baseline (raw + calendar) | 32.97217 | 16.19397 | 27.59912 |
| + network state/rates (PM10, CO) | 29.11627 | 14.45253 | 25.62529 |
| + local history (short+med+long) | 28.61590 | 14.27806 | — |
| + spatial gradients | 29.09116 | 14.10947 | — |
| + rate dynamics | 29.34532 | 14.21072 | — |

Note the last two rows **hurt holdout A** in cumulative order. They are still generated here
— ablation order matters and a family that looks bad stacked last may behave differently
tested standalone. Verify, don't assume.

Ridge behaves differently: it improved on A/B with network features but got *worse* on
Kaggle (31.99 → 32.45). Alpha tuning across 1→300 changed almost nothing, so this is
distribution shift, not regularisation. LightGBM is the more reliable model here.


## 1. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import json
import gc

pd.set_option("display.max_columns", None)

TARGET_COL = "PM2_5_next_hour"

# Base variables available in the corrected dataset
POLLUTANT_COLS = ["PM10", "SO2", "NO2", "CO", "O3"]
WEATHER_COLS   = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
MEASUREMENT_COLS = POLLUTANT_COLS + WEATHER_COLS + ["wd"]

# The two variables with strongest genuine signal — get the full temporal treatment
PRIMARY_COLS = ["PM10", "CO"]

# Registry: family name -> list of generated column names.
# Populated as the notebook runs; exported at the end for ablation.
FEATURE_GROUPS = {}

def register(group, cols):
    """Record which columns belong to which feature family."""
    existing = FEATURE_GROUPS.setdefault(group, [])
    for c in cols:
        if c not in existing:
            existing.append(c)


def downcast(df):
    """float64 -> float32. Halves memory; irrelevant for the precision we need here.
    This notebook builds ~300 features on ~412k rows, which is ~1GB in float64
    before pandas' own copies — downcasting keeps it comfortably in RAM."""
    for c in df.columns:
        if df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
        elif df[c].dtype == "int64":
            df[c] = pd.to_numeric(df[c], downcast="integer")
    gc.collect()
    return df

print("Configuration loaded.")

## 2. Load raw data and verify the corrected files

Fail loudly if someone runs this against the old leaked files.

In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

for df in [train, test]:
    df["observation_timestamp"] = pd.to_datetime(df["observation_timestamp"])

# --- Guard: refuse to run on the old leaked files ---
assert "current_PM2_5" not in train.columns, (
    "current_PM2_5 present — you are using the OLD leaked train.csv. "
    "Download the corrected files."
)
assert "current_PM2_5" not in test.columns, (
    "current_PM2_5 present — you are using the OLD leaked test.csv."
)
assert TARGET_COL in train.columns, "Target missing from train."
assert TARGET_COL not in test.columns, "Target present in test — wrong file."

# Separate target BEFORE any feature engineering
y = train[TARGET_COL].copy()

train_features = train.drop(columns=[TARGET_COL]).copy()
test_features  = test.copy()

# Keep ids aside for submission alignment; never use as a feature
train_ids = train_features["id"].copy()
test_ids  = test_features["id"].copy()
train_features.drop(columns=["id"], inplace=True)
test_features.drop(columns=["id"], inplace=True)

print(f"train_features: {train_features.shape}")
print(f"test_features:  {test_features.shape}")
print(f"y:              {y.shape}  (min {y.min():.1f}, max {y.max():.1f})")
print(f"Train window: {train['observation_timestamp'].min()} -> {train['observation_timestamp'].max()}")
print(f"Test window:  {test['observation_timestamp'].min()} -> {test['observation_timestamp'].max()}")

## 3. Leakage audit (re-run every time)

Cheap insurance. For every remaining column X, check whether `X(t+1)` reconstructs
`target(t)` — the exact pattern that got `current_PM2_5` removed.

**Interpretation:** `exact_match_rate` near 1.0 = leak, stop and investigate.
PM10 at ~0.23 is the known PM10==PM2.5 sensor co-occurrence, not a leak.

In [ ]:
def leakage_audit(train_df, cols):
    ts = train_df.sort_values(["station", "observation_timestamp"]).copy()
    ts["ts_plus_1h"] = ts["observation_timestamp"] + pd.Timedelta(hours=1)
    merged = ts.merge(
        ts[["station", "observation_timestamp"] + cols],
        left_on=["station", "ts_plus_1h"],
        right_on=["station", "observation_timestamp"],
        suffixes=("", "_next"), how="left",
    )
    rows = {}
    for col in cols:
        valid = merged[[TARGET_COL, f"{col}_next"]].dropna()
        if len(valid):
            rows[col] = {
                "n": len(valid),
                "exact_match_rate": np.isclose(
                    valid[TARGET_COL], valid[f"{col}_next"], atol=1e-6).mean(),
                "corr": valid[TARGET_COL].corr(valid[f"{col}_next"]),
            }
    if not rows:
        return pd.DataFrame(columns=["n", "exact_match_rate", "corr"])
    return pd.DataFrame(rows).T.sort_values("exact_match_rate", ascending=False)


audit = leakage_audit(train, POLLUTANT_COLS + WEATHER_COLS)
print(audit)

if len(audit):
    leaky = audit[audit["exact_match_rate"] > 0.9]
    assert len(leaky) == 0, f"POSSIBLE LEAK — investigate before modelling:\n{leaky}"
    print("\nNo leak-like columns detected (all exact_match_rate < 0.9).")
else:
    print("\nWARNING: audit found no consecutive hourly pairs — check the data.")

## 4. Combined timeline

Train and test predictor timelines are concatenated so that lag/rolling features for early
test rows can legitimately reach back into train history. This is **safe**: only predictor
columns are used, never the target. All these columns exist in both files, so anything
computed here is genuinely available at prediction time.

`_is_train` marks the split; it is removed before export.

In [ ]:
train_features["_is_train"] = 1
test_features["_is_train"]  = 0

full = pd.concat([train_features, test_features], ignore_index=True)
full = full.sort_values(["station", "observation_timestamp"]).reset_index(drop=True)

print(f"Combined: {full.shape}")
print(f"Train rows: {(full['_is_train'] == 1).sum()}, test rows: {(full['_is_train'] == 0).sum()}")

# Structural checks
dupes = full.duplicated(subset=["station", "observation_timestamp"]).sum()
print(f"Duplicate (station, timestamp): {dupes}")
assert dupes == 0, "Duplicate station-hours found — investigate before proceeding."

gaps = full.groupby("station")["observation_timestamp"].diff().dt.total_seconds() / 3600
print(f"\nStation-hour gap distribution (hours):")
print(gaps.value_counts().head(5))
print(f"Non-1h gaps: {(gaps.dropna() != 1).sum()} "
      f"({(gaps.dropna() != 1).mean()*100:.2f}%) — this is why lags use exact timestamps.")

## 5. Calendar and cyclical time features

Raw calendar columns encode linear time; cyclical sin/cos encodings additionally encode
periodicity (hour 23 is adjacent to hour 0). Both are kept — linear models need the
cyclical form, trees can split on raw integers directly.

In [ ]:
cal_cols = []

full["year"]      = full["observation_timestamp"].dt.year
full["month"]     = full["observation_timestamp"].dt.month
full["day"]       = full["observation_timestamp"].dt.day
full["hour"]      = full["observation_timestamp"].dt.hour
full["dayofweek"] = full["observation_timestamp"].dt.dayofweek
full["dayofyear"] = full["observation_timestamp"].dt.dayofyear
cal_cols += ["year", "month", "day", "hour", "dayofweek", "dayofyear"]

for name, period in [("hour", 24), ("month", 12), ("dayofweek", 7), ("dayofyear", 365)]:
    full[f"{name}_sin"] = np.sin(2 * np.pi * full[name] / period)
    full[f"{name}_cos"] = np.cos(2 * np.pi * full[name] / period)
    cal_cols += [f"{name}_sin", f"{name}_cos"]

# Winter indicator — test period is Sep 2016-Feb 2017, i.e. cold months.
full["is_winter"] = full["month"].isin([11, 12, 1, 2]).astype(int)
full["is_heating_season"] = full["month"].isin([11, 12, 1, 2, 3]).astype(int)
cal_cols += ["is_winter", "is_heating_season"]

register("CALENDAR", cal_cols)
print(f"CALENDAR: {len(cal_cols)} features")

## 6. Wind encoding

`wd` is a compass string ("NNE", "SW"). Encoded as sin/cos so that N and NNW are close.

**Meteorological convention:** `wd` is the direction the wind comes **FROM**. N means air
travels southward. This matters for the upwind features in §13 — do not reverse it.

Wind vector components (`wind_u`, `wind_v`) decompose speed and direction into orthogonal
parts. Note: in the old leaked pipeline these were tested and **rejected** (holdout A got
worse). Regenerated here because that test was on a different feature set — verify before
keeping.

In [ ]:
WD_TO_DEG = {
    "N": 0,   "NNE": 22.5,  "NE": 45,   "ENE": 67.5,
    "E": 90,  "ESE": 112.5, "SE": 135,  "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225,  "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315,  "NNW": 337.5,
}

wind_cols = []
full["wd_deg"] = full["wd"].map(WD_TO_DEG)
full["wd_sin"] = np.sin(np.deg2rad(full["wd_deg"]))
full["wd_cos"] = np.cos(np.deg2rad(full["wd_deg"]))
wind_cols += ["wd_sin", "wd_cos"]

# Wind vector components (speed x direction)
full["wind_u"] = full["WSPM"] * full["wd_sin"]
full["wind_v"] = full["WSPM"] * full["wd_cos"]
wind_cols += ["wind_u", "wind_v"]

# Calm-air indicator: stagnation concentrates pollution
full["is_calm"] = (full["WSPM"] < 1.0).astype(int)
wind_cols += ["is_calm"]

register("WIND_ENCODING", wind_cols)
print(f"WIND_ENCODING: {len(wind_cols)} features")

## 7. Missingness indicators and quality flags

**Missingness indicators** tell the model *that* a value was absent, separately from
whatever the imputer substitutes. Trees handle NaN natively; linear models do not, so the
flag carries information the imputed value destroys.

**Quality flags** encode sensor artefacts that survive in the corrected data. Note the
PM2.5-based flags from the old pipeline (`pm25_floor_flag`, `pm10_equals_pm25`,
`pm25_gt_pm10`) **cannot be rebuilt** — they all required `current_PM2_5`.

In [ ]:
miss_cols, qual_cols = [], []

for col in MEASUREMENT_COLS:
    name = f"{col}_missing"
    full[name] = full[col].isna().astype(int)
    miss_cols.append(name)

# Sensor limits — CO floor/ceiling were documented instrument bounds
full["co_floor"]    = (full["CO"] == 100).astype(int)
full["co_ceiling"]  = (full["CO"] == 10000).astype(int)
full["o3_extreme"]  = (full["O3"] > 400).astype(int)
full["rain_active"] = (full["RAIN"] > 0).astype(int)
qual_cols += ["co_floor", "co_ceiling", "o3_extreme", "rain_active"]

register("MISSINGNESS", miss_cols)
register("QUALITY_FLAGS", qual_cols)
print(f"MISSINGNESS: {len(miss_cols)} | QUALITY_FLAGS: {len(qual_cols)}")

## 8. Physical derived features

Domain knowledge about pollutant/weather relationships:

- **`TEMP_DEWP_diff`** (dew point depression) — atmospheric dryness. Low values mean high
  relative humidity, which drives particulate formation and hygroscopic growth.
- **`NO2_CO_ratio`, `SO2_CO_ratio`** — all combustion tracers; ratios distinguish
  vehicle-dominated from coal/industrial-dominated pollution.
- **`PM10_CO_ratio`** — coarse particulate relative to combustion tracer; separates dust
  resuspension episodes from combustion episodes.

In [ ]:
phys_cols = []

full["TEMP_DEWP_diff"] = full["TEMP"] - full["DEWP"]
full["NO2_CO_ratio"]   = full["NO2"] / full["CO"].replace(0, np.nan)
full["SO2_CO_ratio"]   = full["SO2"] / full["CO"].replace(0, np.nan)
full["SO2_NO2_ratio"]  = full["SO2"] / full["NO2"].replace(0, np.nan)
full["PM10_CO_ratio"]  = full["PM10"] / full["CO"].replace(0, np.nan)
full["NO2_O3_ratio"]   = full["NO2"] / full["O3"].replace(0, np.nan)
phys_cols += ["TEMP_DEWP_diff", "NO2_CO_ratio", "SO2_CO_ratio",
              "SO2_NO2_ratio", "PM10_CO_ratio", "NO2_O3_ratio"]

# Relative humidity proxy (Magnus approximation)
full["rh_proxy"] = 100 * (
    np.exp((17.625 * full["DEWP"]) / (243.04 + full["DEWP"])) /
    np.exp((17.625 * full["TEMP"]) / (243.04 + full["TEMP"]))
)
phys_cols += ["rh_proxy"]

register("PHYSICAL", phys_cols)
print(f"PHYSICAL: {len(phys_cols)} features")

## 9. Exact-timestamp temporal lags

**Critical — do NOT use `groupby("station")["col"].shift(n)`.** That assumes consecutive
rows are exactly 1 hour apart. They are not: ~0.75% of station-hours are missing. `shift(1)`
across a gap silently returns the value from `t-2h` labelled as `t-1h` — a systematic error
that is invisible in validation.

**Correct approach:** build a `(station, timestamp)` lookup and reindex on
`timestamp - Timedelta(hours=n)`. If that station-hour genuinely doesn't exist, this
returns NaN, which is the honest answer.

Lags are split into SHORT / MEDIUM / LONG families so they can be ablated independently —
in the old pipeline long lags hurt holdout A, but on the corrected data the long family
*helped* both holdouts. Test, don't assume.

In [ ]:
def add_time_lags(df, columns, lags, group_name=None):
    """Exact-timestamp lag builder. Correct across missing station-hours."""
    lookup = (
        df[["station", "observation_timestamp"] + columns]
        .set_index(["station", "observation_timestamp"])
        .sort_index()
    )
    created = []
    for col in columns:
        for lag in lags:
            shifted = df["observation_timestamp"] - pd.Timedelta(hours=lag)
            keys = pd.MultiIndex.from_arrays(
                [df["station"].values, shifted.values],
                names=["station", "observation_timestamp"],
            )
            name = f"{col}_lag_{lag}h"
            df[name] = lookup[col].reindex(keys).to_numpy()
            created.append(name)
    if group_name:
        register(group_name, created)
    return created


# PRIMARY (PM10, CO) get the full lag ladder
short  = add_time_lags(full, PRIMARY_COLS, [1, 2, 3],   "LOCAL_LAGS_SHORT")
medium = add_time_lags(full, PRIMARY_COLS, [6, 12],     "LOCAL_LAGS_MEDIUM")
long_  = add_time_lags(full, PRIMARY_COLS, [24, 48],    "LOCAL_LAGS_LONG")

# Secondary pollutants — shorter ladder
short  += add_time_lags(full, ["SO2", "NO2", "O3"], [1, 3],  "LOCAL_LAGS_SHORT")
medium += add_time_lags(full, ["SO2", "NO2", "O3"], [6, 24], "LOCAL_LAGS_MEDIUM")

# Weather — slow-moving, short ladder only
short  += add_time_lags(full, WEATHER_COLS, [1, 3],  "LOCAL_LAGS_SHORT")
medium += add_time_lags(full, WEATHER_COLS, [6, 24], "LOCAL_LAGS_MEDIUM")

full = downcast(full)
print(f"LOCAL_LAGS_SHORT:  {len(FEATURE_GROUPS['LOCAL_LAGS_SHORT'])}")
print(f"LOCAL_LAGS_MEDIUM: {len(FEATURE_GROUPS['LOCAL_LAGS_MEDIUM'])}")
print(f"LOCAL_LAGS_LONG:   {len(FEATURE_GROUPS['LOCAL_LAGS_LONG'])}")

## 10. Local rolling statistics and rate features

**Rolling stats** give distributional context: PM10 of 100 means something different when
the 24h mean is 20 (spike) versus 150 (relief from a severe episode).

**`closed="left"`** excludes the current timestamp from the window, so the value at `t`
uses only strictly-earlier information. Time-based windows (`"3h"`, not `3`) correctly
handle gaps — a window simply contains fewer points rather than reaching too far back.

**Rate features** are first and second differences. The second difference is curvature:
whether a rise is accelerating or levelling off. In the old pipeline the *local* second
derivative was weak while *regional* change was strong (§12) — both are generated here.

In [ ]:
# ---- Rolling statistics ----
# Assigned directly on a sorted index rather than merged in — a merge per
# window copies the entire (412k x N) frame each time and is the main
# memory spike in this notebook.
roll_cols = []
full = full.sort_values(["station", "observation_timestamp"]).reset_index(drop=True)

for col in PRIMARY_COLS:
    ser = full.set_index("observation_timestamp").groupby("station")[col]
    for window in ["3h", "6h", "12h", "24h"]:
        rolled = ser.rolling(window, closed="left", min_periods=1).agg(
            ["mean", "std", "min", "max"]
        )
        # rolled is indexed (station, timestamp) in the same sorted order as `full`
        rolled = rolled.reset_index(drop=True)
        for stat in ["mean", "std", "min", "max"]:
            name = f"{col}_roll_{stat}_{window}"
            full[name] = rolled[stat].to_numpy(dtype=np.float32)
            roll_cols.append(name)
    del ser
    gc.collect()

register("LOCAL_ROLLING", roll_cols)

# ---- Rate features (built from exact-timestamp lags, so gap-safe) ----
rate_cols = []
for col in PRIMARY_COLS + ["SO2", "NO2", "O3"]:
    for h in [1, 3]:
        lag_name = f"{col}_lag_{h}h"
        if lag_name in full.columns:
            full[f"{col}_change_{h}h"] = full[col] - full[lag_name]
            rate_cols.append(f"{col}_change_{h}h")

for col in PRIMARY_COLS:
    # Percentage change — scale-free momentum
    full[f"{col}_pct_change_1h"] = (
        full[f"{col}_change_1h"] / full[f"{col}_lag_1h"].replace(0, np.nan)
    )
    rate_cols.append(f"{col}_pct_change_1h")

    # Second difference (curvature) — uses lag_1h and lag_2h, both exact-timestamp
    full[f"{col}_second_diff_1h"] = (
        full[col] - 2 * full[f"{col}_lag_1h"] + full[f"{col}_lag_2h"]
    )
    rate_cols.append(f"{col}_second_diff_1h")

    # Deviation from own recent baseline
    full[f"{col}_vs_roll_mean_24h"] = full[col] - full[f"{col}_roll_mean_24h"]
    rate_cols.append(f"{col}_vs_roll_mean_24h")

register("LOCAL_RATES", rate_cols)
full = downcast(full)
print(f"LOCAL_ROLLING: {len(roll_cols)} | LOCAL_RATES: {len(rate_cols)}")
print(f"Frame memory: {full.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 11. Cross-station network state (same timestamp)

These use **other stations' readings at the same timestamp** — not future information. At
prediction time for station X at hour t, every other station's hour-t reading is available.

This is the strongest validated family so far: adding network PM10/CO state and dynamics
moved LightGBM from 27.599 → 25.625 on Kaggle.

Beijing stations are highly spatially correlated (median inter-station r ≈ 0.833 for PM10,
0.796 for CO), so the network state is a strong proxy for the regional pollution field that
`current_PM2_5` used to capture locally.

In [ ]:
net_cols, grad_cols = [], []

for col in POLLUTANT_COLS + ["WSPM", "TEMP"]:
    grp = full.groupby("observation_timestamp")[col]
    full[f"network_{col}_mean"]   = grp.transform("mean")
    full[f"network_{col}_std"]    = grp.transform("std")
    full[f"network_{col}_min"]    = grp.transform("min")
    full[f"network_{col}_max"]    = grp.transform("max")
    full[f"network_{col}_median"] = grp.transform("median")
    net_cols += [f"network_{col}_{s}" for s in ["mean", "std", "min", "max", "median"]]

    # Spatial gradient: how far this station deviates from the city
    full[f"{col}_vs_network_mean"] = full[col] - full[f"network_{col}_mean"]
    grad_cols.append(f"{col}_vs_network_mean")

    # Normalised deviation — scale-free version of the same idea
    full[f"{col}_vs_network_z"] = (
        (full[col] - full[f"network_{col}_mean"]) /
        full[f"network_{col}_std"].replace(0, np.nan)
    )
    grad_cols.append(f"{col}_vs_network_z")

# Network spread — wide range implies a localised episode, narrow implies regional haze
for col in PRIMARY_COLS:
    full[f"network_{col}_range"] = full[f"network_{col}_max"] - full[f"network_{col}_min"]
    net_cols.append(f"network_{col}_range")

register("NETWORK_STATE", net_cols)
register("SPATIAL_GRADIENTS", grad_cols)
full = downcast(full)
print(f"NETWORK_STATE: {len(net_cols)} | SPATIAL_GRADIENTS: {len(grad_cols)}")

## 12. Network history and regional dynamics

**Why these can't be reconstructed from what's already present:** `network_PM10_mean(t)` is
in the feature set, but `network_PM10_mean(t-1h)` is not — and a linear model cannot derive
a lag of a feature from the feature itself.

In the old (leaked) pipeline, leave-one-out ablation found `network_PM25_change_1h` was the
**single strongest dynamic feature**, and `local_vs_network_change` second strongest, while
the *local* second derivative was negligible. The lesson carried forward: what matters is
how the **regional field is moving**, not local curvature. These are the corrected-data
analogues.

Network stats are timestamp-level (identical across stations at a given hour), so the lag
lookup is indexed on timestamp alone.

In [ ]:
NETWORK_LAG_SPEC = [
    (f"network_{col}_{stat}", lags)
    for col in PRIMARY_COLS
    for stat, lags in [("mean", [1, 3, 6, 24]), ("std", [1]),
                        ("min", [1]), ("max", [1])]
]

net_hist_cols, net_rate_cols = [], []

net_lookup = (
    full.drop_duplicates("observation_timestamp")
        .set_index("observation_timestamp")
        .sort_index()
)

for col, lags in NETWORK_LAG_SPEC:
    for lag in lags:
        shifted = full["observation_timestamp"] - pd.Timedelta(hours=lag)
        name = f"{col}_lag_{lag}h"
        full[name] = net_lookup[col].reindex(shifted).to_numpy()
        net_hist_cols.append(name)

# ---- Regional dynamics ----
for col in PRIMARY_COLS:
    m = f"network_{col}_mean"
    full[f"network_{col}_change_1h"] = full[m] - full[f"{m}_lag_1h"]
    full[f"network_{col}_change_3h"] = full[m] - full[f"{m}_lag_3h"]
    full[f"network_{col}_change_24h"] = full[m] - full[f"{m}_lag_24h"]
    net_rate_cols += [f"network_{col}_change_{h}h" for h in [1, 3, 24]]

    # Regional acceleration (second difference of the regional field)
    full[f"network_{col}_second_diff_1h"] = (
        full[m] - 2 * full[f"{m}_lag_1h"] + full[f"{m}_lag_3h"]
    )
    net_rate_cols.append(f"network_{col}_second_diff_1h")

register("NETWORK_HISTORY", net_hist_cols)
register("NETWORK_RATES", net_rate_cols)

# ---- Local vs network divergence ----
lvn_cols = []
for col in PRIMARY_COLS:
    full[f"local_vs_network_{col}_change"] = (
        full[f"{col}_change_1h"] - full[f"network_{col}_change_1h"]
    )
    lvn_cols.append(f"local_vs_network_{col}_change")

register("LOCAL_VS_NETWORK", lvn_cols)
full = downcast(full)
print(f"NETWORK_HISTORY: {len(net_hist_cols)} | NETWORK_RATES: {len(net_rate_cols)} "
      f"| LOCAL_VS_NETWORK: {len(lvn_cols)}")

## 13. Directional / upwind transport features

> ⚠️ **External data:** `STATION_COORDS` uses published station coordinates
> (doi:10.3390/su14095104). Confirm this is permitted under competition rules before
> submitting a model that relies on them.

**Physical motivation:** pollution is advected by wind. An upwind station's current PM10/CO
is partly what will *arrive* at the target station over the next hour. The symmetric network
features treat all 12 stations equally; these weight by actual wind geometry.

**Convention check:** `wd` is where wind comes **FROM**. If `wd = "N"`, air moves southward,
so stations to the **north** are upwind. Getting this backwards silently inverts the
feature — the sign convention below is the one validated in the old pipeline.

**Implementation:** unit vector from target station to each other station, dotted with the
wind-from vector. Stations within a 45° cone (dot > cos 45°) count as upwind. Coverage was
~73.5% of rows previously; the rest have no station in that direction and are NaN-flagged.

In [ ]:
STATION_COORDS = {
    "Aotizhongxin":  (116.397, 39.982), "Changping":     (116.230, 40.217),
    "Dingling":      (116.220, 40.292), "Dongsi":        (116.417, 39.929),
    "Guanyuan":      (116.339, 39.929), "Gucheng":       (116.184, 39.914),
    "Huairou":       (116.628, 40.328), "Nongzhanguan":  (116.461, 39.937),
    "Shunyi":        (116.655, 40.127), "Tiantan":       (116.407, 39.886),
    "Wanliu":        (116.287, 39.987), "Wanshouxigong": (116.352, 39.878),
}

LAT_REF = np.mean([lat for _, lat in STATION_COORDS.values()])
STATION_XY = {
    s: np.array([lon * np.cos(np.radians(LAT_REF)), lat])
    for s, (lon, lat) in STATION_COORDS.items()
}

# Unit direction vector from each station toward every other
station_direction = {}
for tgt, txy in STATION_XY.items():
    station_direction[tgt] = {}
    for oth, oxy in STATION_XY.items():
        if tgt == oth:
            continue
        d = oxy - txy
        station_direction[tgt][oth] = d / np.linalg.norm(d)

def wind_from_vector(deg):
    """Unit vector pointing toward where the wind comes FROM."""
    th = np.radians(deg)
    return np.array([np.sin(th), np.cos(th)])

COS45 = np.cos(np.radians(45))

# Precompute: (target_station, wd_string) -> [(upwind_station, alignment), ...]
UPWIND_MAP = {}
for tgt in STATION_COORDS:
    for wd, deg in WD_TO_DEG.items():
        wvec = wind_from_vector(deg)
        hits = []
        for oth, dvec in station_direction[tgt].items():
            align = float(np.dot(dvec, wvec))
            if align > COS45:
                hits.append((oth, align))
        UPWIND_MAP[(tgt, wd)] = hits

coverage = np.mean([len(v) > 0 for v in UPWIND_MAP.values()])
print(f"Upwind cone coverage across (station, wd) pairs: {coverage:.3f}")

In [ ]:
def build_upwind_features(df, value_cols):
    """Mean / weighted-mean / count of upwind stations' readings at the same timestamp.

    Vectorised per (station, wd) group. All pivots share one common
    (timestamp x station) axis — pivot_table drops all-NaN rows independently
    per column otherwise, which silently misaligns the lookups.
    """
    ts_index = pd.DatetimeIndex(sorted(df["observation_timestamp"].unique()))
    st_index = pd.Index(sorted(df["station"].unique()))

    arrays = {}
    for col in value_cols:
        piv = df.pivot_table(index="observation_timestamp", columns="station", values=col)
        piv = piv.reindex(index=ts_index, columns=st_index)   # <- common axis
        arrays[col] = piv.to_numpy()

    ts_pos = pd.Series(np.arange(len(ts_index)), index=ts_index)
    st_pos = {s: i for i, s in enumerate(st_index)}

    row_ts = df["observation_timestamp"].map(ts_pos).to_numpy()
    n = len(df)

    out = {f"upwind_{c}_{s}": np.full(n, np.nan)
           for c in value_cols for s in ["mean", "weighted"]}
    out["upwind_count"] = np.zeros(n)

    # Group rows by (station, wd) so each group shares the same upwind set
    grouped = df.groupby(["station", "wd"], sort=False).indices

    for (st, wd), row_idx in grouped.items():
        hits = UPWIND_MAP.get((st, wd), [])
        if not hits:
            continue
        cols_idx = np.array([st_pos[o] for o, _ in hits])
        weights = np.array([a for _, a in hits], dtype=float)

        t_idx = row_ts[row_idx].astype(int)
        out["upwind_count"][row_idx] = len(hits)

        for col in value_cols:
            block = arrays[col][np.ix_(t_idx, cols_idx)]      # (rows, upwind stations)
            valid = ~np.isnan(block)
            cnt = valid.sum(axis=1)

            filled = np.where(valid, block, 0.0)
            with np.errstate(invalid="ignore", divide="ignore"):
                mean = np.where(cnt > 0, filled.sum(axis=1) / np.maximum(cnt, 1), np.nan)

                wmat = np.where(valid, weights[None, :], 0.0)
                wsum = wmat.sum(axis=1)
                weighted = np.where(
                    wsum > 0, (filled * wmat).sum(axis=1) / np.maximum(wsum, 1e-12), np.nan
                )

            out[f"upwind_{col}_mean"][row_idx] = mean
            out[f"upwind_{col}_weighted"][row_idx] = weighted

    return out


upwind_out = build_upwind_features(full, PRIMARY_COLS)
upwind_cols = []
for name, arr in upwind_out.items():
    full[name] = arr
    upwind_cols.append(name)

full["upwind_missing"] = (full["upwind_count"] == 0).astype(int)
upwind_cols.append("upwind_missing")

# Upwind history + transport contrast
for col in PRIMARY_COLS:
    m = f"upwind_{col}_mean"
    uw_lookup = full[["station", "observation_timestamp", m]].set_index(
        ["station", "observation_timestamp"]).sort_index()
    for lag in [1, 3]:
        shifted = full["observation_timestamp"] - pd.Timedelta(hours=lag)
        keys = pd.MultiIndex.from_arrays(
            [full["station"].values, shifted.values],
            names=["station", "observation_timestamp"])
        name = f"{m}_lag_{lag}h"
        full[name] = uw_lookup[m].reindex(keys).to_numpy()
        upwind_cols.append(name)

    # Transport contrast: wind speed x (incoming - local). Low marginal correlation
    # but materially useful in the old pipeline — do not drop on correlation alone.
    full[f"WSPM_x_upwind_{col}_contrast"] = full["WSPM"] * (full[m] - full[col])
    upwind_cols.append(f"WSPM_x_upwind_{col}_contrast")

    full[f"upwind_{col}_change_1h"] = full[m] - full[f"{m}_lag_1h"]
    upwind_cols.append(f"upwind_{col}_change_1h")

register("UPWIND", upwind_cols)
full = downcast(full)
print(f"UPWIND: {len(upwind_cols)} features")
print(f"Row-level upwind coverage: {(full['upwind_count'] > 0).mean():.3f}")

## 14. Interaction features

Interactions let a **linear** model capture multiplicative effects. Trees find these
automatically through splits, so for LightGBM they are largely redundant — included anyway
so that Ridge (useful for blending diversity even when individually weaker) has them
available. Regularisation will shrink what isn't useful.

**Important precedent:** `WSPM × (upwind − local)` had a target correlation of only −0.143
yet produced a real leaderboard gain in the old pipeline, while features with r > 0.9 were
often redundant. **Do not select these on marginal correlation** — use ablation.

In [ ]:
inter_cols = []

# Pollutant x weather — weather modulates pollutant persistence
for p in PRIMARY_COLS:
    for w in ["WSPM", "TEMP", "DEWP", "RAIN"]:
        full[f"{p}_x_{w}"] = full[p] * full[w]
        inter_cols.append(f"{p}_x_{w}")

# Pollutant x pollutant — co-occurrence / shared-source effects
for a, b in [("CO", "NO2"), ("CO", "SO2"), ("NO2", "SO2"),
             ("NO2", "O3"), ("PM10", "CO"), ("PM10", "NO2")]:
    full[f"{a}_x_{b}"] = full[a] * full[b]
    inter_cols.append(f"{a}_x_{b}")

# Wind x regional dynamics — how strongly transport drives the regional signal
for col in PRIMARY_COLS:
    full[f"WSPM_x_network_{col}_change"] = full["WSPM"] * full[f"network_{col}_change_1h"]
    full[f"WSPM_x_network_{col}_mean"]   = full["WSPM"] * full[f"network_{col}_mean"]
    inter_cols += [f"WSPM_x_network_{col}_change", f"WSPM_x_network_{col}_mean"]

# Stagnation interactions — calm air concentrates pollution
for col in PRIMARY_COLS:
    full[f"is_calm_x_{col}"] = full["is_calm"] * full[col]
    full[f"is_winter_x_{col}"] = full["is_winter"] * full[col]
    inter_cols += [f"is_calm_x_{col}", f"is_winter_x_{col}"]

# Dew point depression interactions
for col in PRIMARY_COLS:
    full[f"{col}_x_ddd"] = full[col] * full["TEMP_DEWP_diff"]
    inter_cols.append(f"{col}_x_ddd")

register("INTERACTIONS", inter_cols)
full = downcast(full)
print(f"INTERACTIONS: {len(inter_cols)} features")

## 15. Raw feature registration and final assembly

`station` is kept as a categorical column — LightGBM handles it natively, Ridge needs
one-hot encoding downstream. `wd` (raw string) and helper columns are dropped since
`wd_sin`/`wd_cos` supersede them.

In [ ]:
register("RAW", POLLUTANT_COLS + WEATHER_COLS)
register("STATION", ["station"])

# Drop helpers that are not predictors
DROP_ALWAYS = ["wd", "wd_deg"]
full.drop(columns=[c for c in DROP_ALWAYS if c in full.columns], inplace=True)

# Split back out
train_out = full[full["_is_train"] == 1].drop(columns=["_is_train"]).copy()
test_out  = full[full["_is_train"] == 0].drop(columns=["_is_train"]).copy()

# Restore original row order so y stays aligned
train_out = train_out.sort_values("observation_timestamp").reset_index(drop=True)
test_out  = test_out.sort_values("observation_timestamp").reset_index(drop=True)

print(f"train_out: {train_out.shape}")
print(f"test_out:  {test_out.shape}")

# Column parity check
only_train = set(train_out.columns) - set(test_out.columns)
only_test  = set(test_out.columns) - set(train_out.columns)
print(f"Columns only in train (expect empty): {only_train}")
print(f"Columns only in test  (expect empty): {only_test}")
assert not only_train and not only_test, "Train/test column mismatch."

## 16. Re-align the target

`full` was sorted by `(station, timestamp)` for the lag builders, so `y` — which follows the
original `train.csv` row order — is **no longer aligned**. Rebuild the alignment explicitly
via the id column rather than assuming order survived.

This is the single most likely place for a silent bug, so it is asserted rather than trusted.

In [ ]:
# Rebuild alignment through ids
train_key = train[["id", "station", "observation_timestamp", TARGET_COL]].copy()

train_out_keyed = train_out.merge(
    train_key, on=["station", "observation_timestamp"], how="left", validate="one_to_one"
)

assert train_out_keyed[TARGET_COL].isna().sum() == 0, (
    "Target failed to align — some train rows did not match back on (station, timestamp)."
)

y_aligned = train_out_keyed[TARGET_COL].copy()
train_ids_aligned = train_out_keyed["id"].copy()

X_train = train_out_keyed.drop(columns=[TARGET_COL, "id", "observation_timestamp"])

test_keyed = test_out.merge(
    test[["id", "station", "observation_timestamp"]],
    on=["station", "observation_timestamp"], how="left", validate="one_to_one"
)
test_ids_aligned = test_keyed["id"].copy()
X_test = test_keyed.drop(columns=["id", "observation_timestamp"])

assert len(X_train) == len(y_aligned) == len(train), "Train row count changed."
assert len(X_test) == len(test), "Test row count changed."
assert list(X_train.columns) == list(X_test.columns), "Feature order mismatch."

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y:       {y_aligned.shape}")
print("Alignment verified.")

## 17. Feature summary and missingness report

Sanity-check before export. High missingness is expected for long lags (rows near the start
of the record) and for CO-derived features (CO has ~4.4% base missingness that compounds
through lags). Anything at 100% missing is a bug — investigate rather than export.

In [ ]:
numeric_cols = [c for c in X_train.columns if X_train[c].dtype != object]
cat_cols = [c for c in X_train.columns if X_train[c].dtype == object]

print(f"Total features: {X_train.shape[1]}  ({len(numeric_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Categorical: {cat_cols}\n")

print("Features per family:")
for g, cols in sorted(FEATURE_GROUPS.items()):
    present = [c for c in cols if c in X_train.columns]
    print(f"  {g:<22} {len(present):>4}")

miss = X_train[numeric_cols].isna().mean().sort_values(ascending=False)
print(f"\nTop 15 by missingness:")
print(miss.head(15).round(4))

fully_missing = miss[miss == 1.0]
assert len(fully_missing) == 0, f"Fully-missing features (bug): {list(fully_missing.index)}"

print(f"\nFeatures with >20% missing: {(miss > 0.20).sum()}")
print(f"Features with no missing:   {(miss == 0).sum()}")

## 18. Export

Four files:

- **`features_train.parquet`** — engineered training features, aligned to `y.parquet`
- **`features_test.parquet`** — same feature set for test rows
- **`y.parquet`** — target, plus `id` for traceability
- **`feature_groups.json`** — family → column mapping, **use this for ablation**

### For whoever does feature selection

```python
import pandas as pd, json

X_train = pd.read_parquet("features_train.parquet")
X_test  = pd.read_parquet("features_test.parquet")
y       = pd.read_parquet("y.parquet")["PM2_5_next_hour"]
groups  = json.load(open("feature_groups.json"))

# Ablate by family
cols = groups["RAW"] + groups["CALENDAR"] + groups["NETWORK_STATE"]
model.fit(X_train[cols], y)
```

**Validation protocol — do not use random CV.** This is time-series station-hour data;
random folds leak future information. Use the two temporal holdouts:

```python
# Holdout A (seasonal analogue): train pre-Sep 2015, validate Sep 2015 - Feb 2016
# Holdout B (recency):           train pre-Jul 2016, validate Jul - Aug 2016
```

A feature family only counts as a win if it improves **both** holdouts. Several families
here improve one and hurt the other — that is exactly what the ablation is for.

**Also:** clip predictions at 0 before submitting. PM2.5 cannot be negative and Ridge
extrapolates past zero (this was worth ~0.4 RMSE on the baseline).

In [ ]:
X_train.to_parquet("features_train.parquet", index=False)
X_test.to_parquet("features_test.parquet", index=False)

pd.DataFrame({
    "id": train_ids_aligned.values,
    "PM2_5_next_hour": y_aligned.values,
}).to_parquet("y.parquet", index=False)

pd.DataFrame({"id": test_ids_aligned.values}).to_parquet("test_ids.parquet", index=False)

# Only export columns that actually survived
groups_export = {
    g: [c for c in cols if c in X_train.columns]
    for g, cols in FEATURE_GROUPS.items()
}
groups_export = {g: c for g, c in groups_export.items() if c}

# Coverage check: every feature should belong to at least one family
assigned = {c for cols in groups_export.values() for c in cols}
unassigned = [c for c in X_train.columns if c not in assigned]
if unassigned:
    groups_export["UNASSIGNED"] = unassigned
    print(f"Note: {len(unassigned)} features not in a named family -> UNASSIGNED")

with open("feature_groups.json", "w") as f:
    json.dump(groups_export, f, indent=2)

print("=" * 60)
print("  Export complete")
print("=" * 60)
print(f"  features_train.parquet   {X_train.shape}")
print(f"  features_test.parquet    {X_test.shape}")
print(f"  y.parquet                {y_aligned.shape}")
print(f"  test_ids.parquet         {test_ids_aligned.shape}")
print(f"  feature_groups.json      {len(groups_export)} families")
print("=" * 60)
for g, cols in sorted(groups_export.items()):
    print(f"  {g:<22} {len(cols):>4}")
print("=" * 60)